# multivon-eval — RAG Evaluation

[![PyPI](https://img.shields.io/pypi/v/multivon-eval.svg)](https://pypi.org/project/multivon-eval)
[![Docs](https://img.shields.io/badge/docs-evaldocs.multivon.ai-violet)](https://evaldocs.multivon.ai)
[![GitHub](https://img.shields.io/badge/github-multivon--ai%2Fmultivon--eval-black)](https://github.com/multivon-ai/multivon-eval)

**multivon-eval** includes a full suite of RAG evaluators — covering both the generation stage (faithfulness, hallucination, relevance) and the retrieval stage (context precision, recall).

| Part | What |
|------|------|
| 1 | Core generation eval — Faithfulness, Hallucination, Relevance |
| 2 | Retrieval quality — ContextPrecision, ContextRecall |
| 3 | Experiment comparison — compare two retrieval strategies |
| 4 | Plain-English checks for domain-specific criteria |
| 5 | CI integration — fail on regression |

Docs: [evaldocs.multivon.ai](https://evaldocs.multivon.ai) · GitHub: [multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

In [ ]:
!pip install 'multivon-eval>=0.7.0' -q

## API key setup

Set one key here — used both to call the model under test and to run the LLM judge.

multivon-eval uses the key for two things: (1) calling your model via the built-in adapters, and (2) the LLM judge that scores outputs. Both default to the smallest/cheapest model available.

In [ ]:
import os

# Set ONE of these
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # recommended — uses claude-haiku-4-5
# os.environ["OPENAI_API_KEY"]   = "sk-..."       # uses gpt-4o-mini

---
## Part 1 — Core generation eval

Evaluate whether the model's generated answer is **faithful** to the provided context, **relevant** to the question, and free of **hallucinations**.

We use a realistic customer support knowledge base: Acme Corp's return policy.

In [ ]:
# Acme Corp return policy — our knowledge base
ACME_RETURN_POLICY = """
Acme Corp Return Policy

Standard items may be returned within 30 days of purchase for a full refund.
Electronics and appliances have a shorter return window of 14 days from the date of purchase.
Sale items and clearance merchandise are final sale and cannot be returned or exchanged.
Refunds are processed within 5-7 business days after the returned item is received.
To initiate a return, visit acme.com/returns or call our support line at 1-800-555-0100.
Items must be in original condition with tags attached and original packaging.
"""

In [ ]:
import anthropic

client = anthropic.Anthropic()

def rag_model(question: str) -> str:
    """Answer ONLY from the knowledge base context. Say 'I don't know' if the
    answer is not present in the context."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system=(
            "You are a customer support agent for Acme Corp. "
            "Answer the user's question using ONLY the information in the context below. "
            "If the answer is not in the context, say exactly: I don't know.\n\n"
            f"Context:\n{ACME_RETURN_POLICY}"
        ),
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

In [ ]:
from multivon_eval import EvalSuite, EvalCase, Faithfulness, Hallucination, Relevance

suite = EvalSuite.for_rag()
suite.add_check("Response should not introduce information not in the knowledge base")

suite.add_cases([
    EvalCase(
        input="What is the return window for standard items?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="Can I return electronics after 20 days?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="How long does it take to receive my refund?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="Can I return a sale item I bought last week?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        # Out-of-context question — the model should say it doesn't know
        input="What was Acme Corp's revenue last quarter?",
        context=ACME_RETURN_POLICY,
    ),
])

report = suite.run(rag_model)
print(f"Pass rate: {report.pass_rate:.0%}  ({report.total} cases)")

In [ ]:
# Scores broken down by evaluator
print("Scores by evaluator:")
for name, score in report.scores_by_evaluator().items():
    print(f"  {name}: {score:.2f}")

# Inspect any failures
if report.failed_cases:
    print(f"\nFailed cases: {len(report.failed_cases)}")
    for cr in report.failed_cases:
        print(f"\n  Input:  {cr.case_input}")
        print(f"  Output: {cr.actual_output[:200]}")
        for r in cr.results:
            if not r.passed:
                print(f"    ✗ {r.evaluator} — {r.reason[:200]}")

---
## Part 2 — Retrieval quality

`ContextPrecision` and `ContextRecall` evaluate the **retrieval stage** independently from generation.

- **ContextPrecision** — Are the retrieved chunks relevant to the question? High precision means low noise.
- **ContextRecall** — Does the retrieved context contain everything needed to answer? Low recall means the model is missing key information.

Pass `context` as a `list[str]` (one element per chunk) to evaluate at the chunk level.

In [ ]:
from multivon_eval import EvalSuite, EvalCase, ContextPrecision, ContextRecall

# Simulate a retriever that returns 4 chunks — 2 relevant, 2 noise
chunks_with_noise = [
    # Relevant chunks
    "Standard items may be returned within 30 days of purchase for a full refund.",
    "To initiate a return, visit acme.com/returns or call 1-800-555-0100.",
    # Noise chunks — retrieved but not relevant to the question
    "Acme Corp was founded in 1987 and is headquartered in San Francisco, CA.",
    "Acme Corp offers free shipping on orders over $50 within the continental US.",
]

suite_precision = EvalSuite("Retrieval Precision")
suite_precision.add_evaluators(ContextPrecision(), ContextRecall())
suite_precision.add_cases([
    EvalCase(
        input="How do I return an item and what is the return window?",
        context=chunks_with_noise,  # list[str] — one chunk per element
    ),
])

report_precision = suite_precision.run(rag_model)
print("Precision / Recall with noisy retrieval:")
for name, score in report_precision.scores_by_evaluator().items():
    print(f"  {name}: {score:.2f}")

In [ ]:
# Low-recall scenario — the relevant chunk is MISSING from the retrieved context
# The retriever fetched chunks about shipping and founding, but not the refund timeline
chunks_missing_key_info = [
    "Acme Corp was founded in 1987 and is headquartered in San Francisco, CA.",
    "Acme Corp offers free shipping on orders over $50 within the continental US.",
    "Acme Corp's loyalty programme gives 1 point per dollar spent.",
]

suite_recall = EvalSuite("Retrieval Recall")
suite_recall.add_evaluators(ContextPrecision(), ContextRecall())
suite_recall.add_cases([
    EvalCase(
        input="How long does a refund take to process?",
        context=chunks_missing_key_info,  # missing the refund timeline chunk
    ),
])

report_recall = suite_recall.run(rag_model)
print("Precision / Recall when key chunk is missing:")
for name, score in report_recall.scores_by_evaluator().items():
    print(f"  {name}: {score:.2f}")
print("\nExpect low recall — the context doesn't contain the answer.")

---
## Part 3 — Experiment comparison

Use `Experiment` to compare two retrieval strategies head-to-head.

- **Strategy A** (narrow): returns only the single most relevant chunk — high precision, may miss supporting context.
- **Strategy B** (broad): returns the top 3 chunks — higher recall, but more noise.

`exp.compare()` gives a side-by-side pass rate and per-evaluator breakdown.

In [ ]:
# Simulated retrieval strategies — replace with your real retriever calls

CHUNK_RETURN_WINDOW = "Standard items may be returned within 30 days. Electronics within 14 days."
CHUNK_SALE_ITEMS    = "Sale items and clearance merchandise are final sale and cannot be returned."
CHUNK_REFUND_TIME   = "Refunds are processed within 5-7 business days after the item is received."
CHUNK_CONTACT       = "To initiate a return, visit acme.com/returns or call 1-800-555-0100."
CHUNK_CONDITION     = "Items must be in original condition with tags attached and original packaging."

def rag_v1(question: str) -> str:
    """Strategy A — narrow retrieval: top-1 chunk only."""
    # Simulates a retriever that returns only the single highest-ranked chunk
    narrow_context = {
        "What is the return window for standard items?": CHUNK_RETURN_WINDOW,
        "Can I return electronics after 20 days?":       CHUNK_RETURN_WINDOW,
        "How long does it take to receive my refund?":   CHUNK_REFUND_TIME,
        "Can I return a sale item I bought last week?":  CHUNK_SALE_ITEMS,
    }.get(question, "No relevant information found.")

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system=(
            "You are a customer support agent. Answer using ONLY the context. "
            "If the answer is not in the context, say: I don't know.\n\n"
            f"Context:\n{narrow_context}"
        ),
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text


def rag_v2(question: str) -> str:
    """Strategy B — broad retrieval: top-3 chunks."""
    # Simulates a retriever that returns the top 3 chunks
    broad_context = {
        "What is the return window for standard items?": "\n".join([
            CHUNK_RETURN_WINDOW, CHUNK_CONTACT, CHUNK_CONDITION,
        ]),
        "Can I return electronics after 20 days?": "\n".join([
            CHUNK_RETURN_WINDOW, CHUNK_SALE_ITEMS, CHUNK_CONTACT,
        ]),
        "How long does it take to receive my refund?": "\n".join([
            CHUNK_REFUND_TIME, CHUNK_RETURN_WINDOW, CHUNK_CONTACT,
        ]),
        "Can I return a sale item I bought last week?": "\n".join([
            CHUNK_SALE_ITEMS, CHUNK_RETURN_WINDOW, CHUNK_CONTACT,
        ]),
    }.get(question, "No relevant information found.")

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system=(
            "You are a customer support agent. Answer using ONLY the context. "
            "If the answer is not in the context, say: I don't know.\n\n"
            f"Context:\n{broad_context}"
        ),
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

In [ ]:
from multivon_eval import EvalSuite, EvalCase, Faithfulness, Hallucination, Relevance

shared_cases = [
    EvalCase(
        input="What is the return window for standard items?",
        context=CHUNK_RETURN_WINDOW,
    ),
    EvalCase(
        input="Can I return electronics after 20 days?",
        context=CHUNK_RETURN_WINDOW,
    ),
    EvalCase(
        input="How long does it take to receive my refund?",
        context=CHUNK_REFUND_TIME,
    ),
    EvalCase(
        input="Can I return a sale item I bought last week?",
        context=CHUNK_SALE_ITEMS,
    ),
]

suite_a = EvalSuite.for_rag()
suite_a.add_cases(shared_cases)
report_a = suite_a.run(rag_v1)

suite_b = EvalSuite.for_rag()
suite_b.add_cases(shared_cases)
report_b = suite_b.run(rag_v2)

print(f"Strategy A (chunk-1) pass rate: {report_a.pass_rate:.0%}")
print(f"Strategy B (chunk-3) pass rate: {report_b.pass_rate:.0%}")

In [ ]:
from multivon_eval import Experiment

exp = Experiment("RAG chunk size comparison")
# `record(report, run_id=...)` stores a run and returns the resolved run_id
run_a_id = exp.record(report_a, run_id="chunk-1")
run_b_id = exp.record(report_b, run_id="chunk-3")

# `compare(a, b)` prints a side-by-side diff with p-values
exp.compare(run_a_id, run_b_id)

---
## Part 4 — Plain-English checks for domain-specific criteria

`add_check()` lets you express quality criteria that standard evaluators don't cover — things specific to your domain, brand voice, or compliance requirements.

The SDK auto-generates yes/no questions from your criterion. After `prepare()`, inspect `resolved_questions` to see exactly what will be evaluated — and pin them for reproducible CI runs.

In [ ]:
from multivon_eval import EvalSuite, EvalCase

suite_domain = EvalSuite.for_rag()

# Domain-specific criteria that Faithfulness/Hallucination alone won't catch
suite_domain.add_check(
    "Response should mention the specific return window in days"
)
suite_domain.add_check(
    "Response should not make up contact information such as phone numbers or URLs"
)

suite_domain.add_cases([
    EvalCase(
        input="What is the return window for standard items?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="How do I start a return?",
        context=ACME_RETURN_POLICY,
    ),
])

# Inspect the questions the SDK will ask the LLM judge.
# CheckEvaluator.prepare() resolves the criterion → questions via the judge.
# (It's also called automatically inside suite.run(), so this preview step
# is optional — just useful when you want to pin questions for CI.)
for ev in suite_domain.evaluators:
    if hasattr(ev, "criterion"):
        ev.prepare()
        print(f"Criterion: {ev.criterion}")
        for i, q in enumerate(ev.resolved_questions or [], 1):
            print(f"  {i}. {q}")
        print()

In [ ]:
report_domain = suite_domain.run(rag_model)

print(f"Pass rate: {report_domain.pass_rate:.0%}")
print("\nScores by evaluator:")
for name, score in report_domain.scores_by_evaluator().items():
    print(f"  {name}: {score:.2f}")

---
## Part 5 — CI integration

Block deploys when eval quality drops below a threshold. The pattern:

1. Set `fail_threshold` — the suite fails if `pass_rate < threshold`.
2. Save the report as JSON for diffing between runs.
3. Exit with code 1 if the report failed — CI picks this up as a failed check.

In [ ]:
import sys
from multivon_eval import EvalSuite, EvalCase

suite_ci = EvalSuite.for_rag()
suite_ci.add_check("Response should mention the specific return window in days")
suite_ci.add_check("Response should not make up contact information")

suite_ci.add_cases([
    EvalCase(
        input="What is the return window for standard items?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="Can I return electronics after 20 days?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="How long does it take to receive my refund?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="Can I return a sale item I bought last week?",
        context=ACME_RETURN_POLICY,
    ),
    EvalCase(
        input="What was Acme Corp's revenue last quarter?",
        context=ACME_RETURN_POLICY,
    ),
])

# fail_threshold=0.8 means the suite fails if pass_rate < 80%
report_ci = suite_ci.run(rag_model, fail_threshold=0.8)

print(f"Pass rate:  {report_ci.pass_rate:.0%}")
print(f"Avg score:  {report_ci.avg_score:.2f}")
print(f"Passed:     {report_ci.passed}")

In [ ]:
# Save for CI artifact storage and regression diffing
report_ci.save_json("rag_report.json")   # machine-readable, diff across runs
report_ci.save_html("rag_report.html")   # self-contained HTML with per-case breakdown

print("Saved: rag_report.json  rag_report.html")

In [ ]:
# In a CI script (not a notebook), use sys.exit to fail the pipeline:
#
#   if not report_ci.passed:
#       print(f"RAG eval failed: pass rate {report_ci.pass_rate:.0%} < 80% threshold")
#       sys.exit(1)
#
# In the notebook we just print the outcome:
if report_ci.passed:
    print("Eval passed — safe to deploy.")
else:
    print(
        f"Eval FAILED — pass rate {report_ci.pass_rate:.0%} is below the 80% threshold.\n"
        "Review rag_report.html before merging."
    )

---
## Next steps

- **Docs:** [evaldocs.multivon.ai](https://evaldocs.multivon.ai)
- **Agent evaluators:** tool call accuracy, trajectory efficiency, plan quality → [Agent Evaluators](https://evaldocs.multivon.ai/evaluators/agent)
- **Experiment tracking:** compare runs with p-values → [Experiment Tracking](https://evaldocs.multivon.ai/guides/experiments)
- **CI/CD integration:** block deploys on regression → [CI/CD Guide](https://evaldocs.multivon.ai/guides/ci-cd)
- **GitHub:** [github.com/multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

**Found a bug or want a feature?** Open an issue: [github.com/multivon-ai/multivon-eval/issues](https://github.com/multivon-ai/multivon-eval/issues)